# CodeCrafters webinar evaluation: CRISP-DM mixed-methods analysis

This notebook is the execution entry point for the reproducible analysis pipeline. It follows CRISP-DM: business understanding, data understanding, data preparation, descriptive modeling/analysis, evaluation, and deployment. Run it with the local Conda/Jupyter kernel when you need generated outputs to appear in this Git working tree. The raw CSV is read-only; derived tables and figures are written under `ANALYSIS/outputs/`.

## Colab setup

The preferred execution mode is the local Conda/Jupyter kernel because it shares this VS Code workspace and writes `ANALYSIS/outputs/` directly into the repository. A Colab kernel remains supported as a remote fallback, but its files must be exported back before they can be committed.

In [ ]:
import importlib.util

required_packages = ['pandas', 'numpy', 'matplotlib', 'seaborn']
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing_packages:
    raise ImportError(f'Missing packages in the active Conda/Jupyter environment: {missing_packages}. Install them in the active environment only if necessary.')
print('Required analysis packages are available in the active kernel; no notebook installation was run.')

In [ ]:
from pathlib import Path
import sys
import subprocess
import zipfile

def is_project_root(candidate):
    analysis_dir = candidate / 'ANALYSIS'
    return (analysis_dir / 'src').is_dir() and (
        (analysis_dir / 'data').is_dir() or (candidate / 'data').is_dir()
    )

def find_project_root(search_roots):
    for search_root in search_roots:
        if not search_root.exists():
            continue
        upward_candidates = [search_root, *search_root.parents]
        for candidate in upward_candidates:
            if is_project_root(candidate):
                return candidate
        for analysis_dir in search_root.rglob('ANALYSIS'):
            candidate = analysis_dir.parent
            if is_project_root(candidate):
                return candidate
    return None

# Local Conda/Jupyter is the default. This path is optional when the notebook
# is opened from inside the repository; the upward search normally finds it.
LOCAL_PROJECT_ROOT = None

# The Git/ZIP settings are used only when the active kernel is genuine Colab.
REPOSITORY_URL = 'https://github.com/adriaanamd/operation-brea-olla-iv-a.git'
REPOSITORY_REF = 'master'
REMOTE_CLONE_DIR = Path('/content/organized-webminar-manuscript')

if LOCAL_PROJECT_ROOT is not None:
    PROJECT_ROOT = Path(LOCAL_PROJECT_ROOT).expanduser().resolve()
else:
    PROJECT_ROOT = find_project_root([Path.cwd()])

is_colab_runtime = False
try:
    import google.colab  # type: ignore
    is_colab_runtime = True
except ImportError:
    pass

if PROJECT_ROOT is None and is_colab_runtime:
    # Search the remote Colab filesystem only after local discovery fails.
    PROJECT_ROOT = find_project_root([Path('/content')])
    if PROJECT_ROOT is None:
        clone_command = [
            'git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF,
            REPOSITORY_URL, str(REMOTE_CLONE_DIR),
        ]
        clone_result = subprocess.run(clone_command, capture_output=True, text=True)
        if clone_result.returncode == 0:
            PROJECT_ROOT = REMOTE_CLONE_DIR
        else:
            print('Git clone failed; upload a ZIP of the repository instead.')

if PROJECT_ROOT is None and is_colab_runtime:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            'The Colab kernel is remote, so it cannot see the local VS Code workspace. '
            'Use the local Conda/Jupyter kernel for repository-local outputs, or upload a ZIP. '
            'Do not place a GitHub token or password in this notebook.'
        ) from exc
    print('No manuscript repository found. Upload a ZIP containing ANALYSIS/src and data.')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('Upload a ZIP of the manuscript repository, not only the notebook file.')
    for zip_name in zip_names:
        with zipfile.ZipFile(zip_name) as archive:
            archive.extractall('/content')
    PROJECT_ROOT = find_project_root([Path('/content')])

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'The active kernel cannot locate the manuscript repository. In local Conda/Jupyter, '
        'open the notebook from the repository or set LOCAL_PROJECT_ROOT to the local manuscript folder.'
    )

sys.path.insert(0, str(PROJECT_ROOT))
data_dir = PROJECT_ROOT / 'ANALYSIS' / 'data'
if not data_dir.is_dir():
    data_dir = PROJECT_ROOT / 'data'
data_files = sorted(data_dir.glob('*.csv'))
if len(data_files) != 1:
    raise FileNotFoundError(f'Expected exactly one CSV in ANALYSIS/data; found {len(data_files)}.')
input_path = data_files[0]
print('Project root:', PROJECT_ROOT)
print('Data directory:', data_dir)
print('Input CSV:', input_path.name)
print('Python:', sys.version.split()[0])
print('Execution mode:', 'Colab remote runtime' if is_colab_runtime else 'local Conda/Jupyter')

## CRISP-DM 1 — Business understanding

**Decision context.** The analysis supports decisions about whether the webinar's delivery, materials, technical setup, and learning relevance should be retained or improved in future sessions. The audience is the manuscript team and webinar organizers.

**Primary objective.** Describe post-event participant perceptions using the 13-item Likert matrix, two separate summary ratings, and analyst-coded open-ended feedback.

**Decision questions.** (1) Which experience dimensions were strongest and weakest? (2) How stable are the summaries when the two scale-direction anomalies are normalized? (3) What did participants report learning or valuing? (4) What concrete improvements are supported by both rating patterns and written feedback?

**Success criteria.** Every reported metric has an explicit numerator, denominator, scale rule, and uncertainty or limitation note; every transformation is reproducible; qualitative codes remain separate from quantitative ratings; and no result is interpreted as objective learning, causality, or population-level generalizability.

## CRISP-DM 2 — Data understanding

The source is a desensitized Google Forms CSV. Its grain is one respondent submission. The analysis uses the 13 experience statements as the primary Likert block, keeps overall rating and satisfaction as separate single-item measures, and treats the four open-ended prompts as a qualitative evidence stream. The profile below checks shape, schema, missingness, range validity, identifier uniqueness, and response coverage before interpretation.

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Image, display

from ANALYSIS.src.pipeline.config import (
    PRIMARY_ITEM_KEYS, PRIMARY_ITEM_LABELS, SUMMARY_KEYS, TEXT_KEYS,
)
from ANALYSIS.src.pipeline.io import build_rating_frame, build_text_frame, load_evaluation_csv
from ANALYSIS.src.pipeline.quality import flag_scale_direction_rows, validate_frame

output_dir = PROJECT_ROOT / 'ANALYSIS' / 'outputs'
tables = output_dir / 'tables'
figures = output_dir / 'figures'
for path in (tables, figures, output_dir / 'processed'):
    path.mkdir(parents=True, exist_ok=True)

raw = load_evaluation_csv(input_path)
ratings = build_rating_frame(raw)
text = build_text_frame(raw)
quality = validate_frame(raw, ratings)
timestamp_column = raw.columns[1]
institution_column = raw.columns[5]
text_response_counts = {
    key: int(text[key].ne('').sum()) for key in TEXT_KEYS
}
profile = pd.DataFrame([
    {'check': 'respondent rows', 'value': len(raw), 'definition': 'One row per submitted response'},
    {'check': 'raw columns excluding generated respondent_id', 'value': quality['raw_columns'], 'definition': 'Fields in the supplied CSV'},
    {'check': 'primary Likert items', 'value': len(PRIMARY_ITEM_KEYS), 'definition': 'Normalized 1-5 experience statements'},
    {'check': 'summary rating items', 'value': len(SUMMARY_KEYS), 'definition': 'Overall rating and satisfaction, retained separately'},
    {'check': 'open-ended prompts', 'value': len(TEXT_KEYS), 'definition': 'Qualitative response fields'},
    {'check': 'missing rating cells', 'value': quality['invalid_rating_cells'], 'definition': 'Missing values in the 15 rating fields'},
    {'check': 'out-of-range rating cells', 'value': quality['out_of_range_cells'], 'definition': 'Values outside the 1-5 response range'},
    {'check': 'duplicate generated respondent IDs', 'value': int(raw['respondent_id'].duplicated().sum()), 'definition': 'Uniqueness check for de-identified row IDs'},
    {'check': 'duplicate timestamps', 'value': int(raw[timestamp_column].duplicated().sum()), 'definition': 'Diagnostic only; timestamp is not used as an identifier'},
])
display(profile)
display(pd.DataFrame([
    {'prompt': key, 'nonblank_responses': count, 'response_pct': round(count / len(text) * 100, 2)}
    for key, count in text_response_counts.items()
]))
schema = pd.DataFrame([
    {'stream': 'metadata', 'fields': f'{timestamp_column}; {institution_column}'},
    {'stream': 'quantitative', 'fields': '13 primary Likert items; overall_rating; satisfaction'},
    {'stream': 'qualitative', 'fields': ', '.join(TEXT_KEYS)},
])
display(schema)
profile.to_csv(tables / 'crispdm_data_profile.csv', index=False)
pd.DataFrame([quality]).to_csv(tables / 'crispdm_validation_summary.csv', index=False)

## CRISP-DM 3 — Data preparation

The reusable pipeline reads the raw file without overwriting it, assigns de-identified row IDs, parses the first valid 1–5 response from each rating cell, and extracts open-ended fields under stable names. The scale-direction rule is deliberately narrow: a respondent is flagged only when all 13 primary items are raw 1 while both summary ratings are at least 4. For flagged rows, only the primary matrix is reverse-scored with `6 - raw`; overall rating and satisfaction remain unchanged as anchors. Thematic coding uses the existing codebook and de-identified coding decisions in `ANALYSIS/src/themes/`.

In [ ]:
from ANALYSIS.src.pipeline.run_analysis import run_analysis

manifest = run_analysis(input_path=input_path, output_dir=output_dir)
normalized = pd.read_csv(output_dir / 'processed' / 'normalized_ratings.csv')
flags = pd.read_csv(tables / 'scale_direction_flags.csv')
audit_path = tables / 'normalization_audit.csv'
audit = pd.read_csv(audit_path) if audit_path.stat().st_size else pd.DataFrame()
likert = pd.read_csv(tables / 'likert_item_summary.csv')
summary = pd.read_csv(tables / 'overall_satisfaction_summary.csv')
themes = pd.read_csv(tables / 'thematic_counts.csv')
matrix = pd.read_csv(tables / 'mixed_methods_evidence_matrix.csv')

print('Rows:', manifest['rows'])
print('Flagged scale-direction rows:', manifest['flagged_rows'])
print('Normalization scope:', manifest['normalization_scope'])
display(flags.loc[flags['flagged']].copy())
display(audit.head(10))

## CRISP-DM 4 — Modeling and analysis

Because this is a one-time, nonexperimental post-event evaluation, modeling is defined here as descriptive and exploratory evidence generation rather than predictive machine learning. The notebook reports distributions, means, medians, dispersion, agreement and top-box rates, bootstrap intervals for means, Wilson intervals for proportions, a descriptive 13-item composite, item rankings, institutional reach, and a speaker-clarity-by-overall-rating cross-tab. These are associative summaries, not causal estimates.

In [ ]:
RANDOM_SEED = 20261012
BOOTSTRAP_REPS = 2000
rng = np.random.default_rng(RANDOM_SEED)

def wilson_interval(successes, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = successes / n
    denominator = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denominator
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denominator
    return (100 * (centre - margin), 100 * (centre + margin))

def bootstrap_mean_interval(values, generator, reps=BOOTSTRAP_REPS):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return (np.nan, np.nan)
    draws = generator.choice(values, size=(reps, values.size), replace=True)
    means = draws.mean(axis=1)
    return tuple(np.quantile(means, [0.025, 0.975]))

def metric_row(frame, key, label, stream):
    values = frame[key].dropna().astype(float).to_numpy()
    n = values.size
    agree_n = int((values >= 4).sum())
    top_box_n = int((values == 5).sum())
    agree_low, agree_high = wilson_interval(agree_n, n)
    mean_low, mean_high = bootstrap_mean_interval(values, rng)
    return {
        'stream': stream, 'item_key': key, 'item_label': label, 'n': int(n),
        'mean': round(float(values.mean()), 4), 'median': float(np.median(values)),
        'sd': round(float(values.std(ddof=1)), 4) if n > 1 else np.nan,
        'min': int(values.min()), 'max': int(values.max()),
        'agree_n_4_or_5': agree_n, 'agree_pct_4_or_5': round(100 * agree_n / n, 2),
        'agree_ci95_low': round(agree_low, 2), 'agree_ci95_high': round(agree_high, 2),
        'top_box_n_5': top_box_n, 'top_box_pct_5': round(100 * top_box_n / n, 2),
        'mean_bootstrap_ci95_low': round(float(mean_low), 4),
        'mean_bootstrap_ci95_high': round(float(mean_high), 4),
    }

metric_rows = [
    metric_row(normalized, key, label, 'primary_likert')
    for key, label in zip(PRIMARY_ITEM_KEYS, PRIMARY_ITEM_LABELS)
] + [
    metric_row(normalized, key, key.replace('_', ' ').title(), 'summary_rating')
    for key in SUMMARY_KEYS
]
quantitative_metrics = pd.DataFrame(metric_rows)
display(quantitative_metrics)
quantitative_metrics.to_csv(tables / 'crispdm_quantitative_metrics.csv', index=False)

In [ ]:
analysis_frame = normalized.copy()
analysis_frame['primary_likert_mean'] = analysis_frame[PRIMARY_ITEM_KEYS].mean(axis=1)
analysis_frame['primary_likert_median'] = analysis_frame[PRIMARY_ITEM_KEYS].median(axis=1)
composite_summary = pd.DataFrame([{
    'metric': '13-item normalized experience composite',
    'n': int(analysis_frame['primary_likert_mean'].notna().sum()),
    'mean': round(float(analysis_frame['primary_likert_mean'].mean()), 4),
    'median': round(float(analysis_frame['primary_likert_mean'].median()), 4),
    'sd': round(float(analysis_frame['primary_likert_mean'].std(ddof=1)), 4),
    'min': round(float(analysis_frame['primary_likert_mean'].min()), 4),
    'max': round(float(analysis_frame['primary_likert_mean'].max()), 4),
    'agree_composite_pct': round(float((analysis_frame['primary_likert_mean'] >= 4).mean() * 100), 2),
}])
display(composite_summary)

institution_values = raw[institution_column].fillna('').astype(str).str.strip().replace('', 'Unspecified/NA')
institution_summary = institution_values.value_counts(dropna=False).rename_axis('institution').reset_index(name='n')
institution_summary['pct'] = (institution_summary['n'] / len(raw) * 100).round(2)
display(institution_summary)

clarity_key = 'item_04'
clarity_labels = {1: '1 - Strongly disagree', 2: '2', 3: '3', 4: '4', 5: '5 - Strongly agree'}
speaker_cross = pd.crosstab(normalized[clarity_key], normalized['overall_rating'])
speaker_cross = speaker_cross.reindex(index=range(1, 6), columns=range(1, 6), fill_value=0)
speaker_cross.index = [clarity_labels[i] for i in speaker_cross.index]
display(speaker_cross)
speaker_summary = (
    normalized.groupby(clarity_key, dropna=False)['overall_rating']
    .agg(n='count', mean_overall='mean', top_box_n=lambda s: int((s == 5).sum()), top_box_pct=lambda s: float((s == 5).mean() * 100))
    .reset_index()
)
speaker_summary['clarity_label'] = speaker_summary[clarity_key].map(clarity_labels)
speaker_summary['top_box_pct'] = speaker_summary['top_box_pct'].round(2)
display(speaker_summary)
speaker_spearman = normalized[[clarity_key, 'overall_rating']].corr(method='spearman').iloc[0, 1]
association_summary = pd.DataFrame([{'comparison': 'speaker clarity vs overall rating', 'n': int(normalized[[clarity_key, 'overall_rating']].dropna().shape[0]), 'spearman_rho': round(float(speaker_spearman), 4), 'interpretation': 'Descriptive ordinal association; not causal.'}])
display(association_summary)

composite_summary.to_csv(tables / 'crispdm_composite_summary.csv', index=False)
institution_summary.to_csv(tables / 'crispdm_institution_summary.csv', index=False)
speaker_cross.to_csv(tables / 'crispdm_speaker_clarity_overall_crosstab.csv')
speaker_summary.to_csv(tables / 'crispdm_speaker_clarity_overall_summary.csv', index=False)
association_summary.to_csv(tables / 'crispdm_association_summary.csv', index=False)

## CRISP-DM 5 — Evaluation and validation

The evaluation stage checks that the analysis is internally consistent and that the normalization did not alter the two summary anchors. It also quantifies sensitivity by comparing raw and normalized primary-item means. These checks validate computation; they do not validate the original respondents' intent or establish psychometric validity.

In [ ]:
raw_item_means = ratings[PRIMARY_ITEM_KEYS].astype(float).mean()
normalized_item_means = normalized[PRIMARY_ITEM_KEYS].astype(float).mean()
normalization_sensitivity = pd.DataFrame({
    'item_key': PRIMARY_ITEM_KEYS,
    'raw_mean': [raw_item_means[key] for key in PRIMARY_ITEM_KEYS],
    'normalized_mean': [normalized_item_means[key] for key in PRIMARY_ITEM_KEYS],
})
normalization_sensitivity['mean_change'] = normalization_sensitivity['normalized_mean'] - normalization_sensitivity['raw_mean']
normalization_sensitivity = normalization_sensitivity.round(4)
display(normalization_sensitivity)
normalization_sensitivity.to_csv(tables / 'crispdm_normalization_sensitivity.csv', index=False)

checks = pd.DataFrame([
    {'check': 'raw and normalized row counts match', 'passed': len(raw) == len(normalized)},
    {'check': 'all rating cells remain in 1-5 range', 'passed': bool(((normalized[PRIMARY_ITEM_KEYS + SUMMARY_KEYS] >= 1) & (normalized[PRIMARY_ITEM_KEYS + SUMMARY_KEYS] <= 5)).all().all())},
    {'check': 'summary ratings unchanged by normalization', 'passed': np.array_equal(ratings[SUMMARY_KEYS].to_numpy(), normalized[SUMMARY_KEYS].to_numpy())},
    {'check': 'manifest row count matches source', 'passed': int(manifest['rows']) == len(raw)},
    {'check': 'flag count matches manifest', 'passed': int(flags['flagged'].sum()) == int(manifest['flagged_rows'])},
    {'check': 'audit rows equal flagged rows times 13', 'passed': len(audit) == int(manifest['flagged_rows']) * len(PRIMARY_ITEM_KEYS)},
])
display(checks)
if not bool(checks['passed'].all()):
    raise AssertionError('At least one reproducibility or normalization check failed.')

## Mixed-methods integration: qualitative evidence

The thematic stream is analyzed at the unit of one respondent answer to one open-ended prompt. Multiple codes are allowed for distinct ideas in one answer. Counts therefore represent coded assignments and respondent coverage, not mutually exclusive participant categories. The codebook and coding decisions are retained as auditable project assets.

In [ ]:
theme_decisions = pd.read_csv(PROJECT_ROOT / 'ANALYSIS' / 'src' / 'themes' / 'coding_decisions.csv')
theme_counts = themes.copy()
theme_counts['respondent_coverage_pct'] = (theme_counts['n_respondents'] / len(raw) * 100).round(2)
theme_counts['assignment_share_pct'] = (theme_counts['n_assignments'] / theme_counts['n_assignments'].sum() * 100).round(2)
display(theme_counts)
theme_by_prompt = pd.crosstab(theme_decisions['source_field'], theme_decisions['code_id'])
display(theme_by_prompt)
theme_counts.to_csv(tables / 'crispdm_theme_counts_with_denominators.csv', index=False)
theme_by_prompt.to_csv(tables / 'crispdm_theme_by_prompt.csv')

In [ ]:
item_plot = quantitative_metrics[quantitative_metrics['stream'] == 'primary_likert'].sort_values('mean')
fig, ax = plt.subplots(figsize=(10, 7))
ax.errorbar(
    item_plot['mean'], item_plot['item_label'],
    xerr=[item_plot['mean'] - item_plot['mean_bootstrap_ci95_low'], item_plot['mean_bootstrap_ci95_high'] - item_plot['mean']],
    fmt='o', color='#2f6690', ecolor='#7a9db8', capsize=3,
)
ax.set_xlim(1, 5)
ax.set_xlabel('Mean normalized rating (1-5), with bootstrap 95% interval')
ax.set_ylabel('')
ax.set_title('Primary Likert item performance')
fig.tight_layout()
fig.savefig(figures / 'crispdm_item_means_with_ci.png', dpi=180)
plt.show()

plt.figure(figsize=(7, 5))
sns.heatmap(speaker_cross, annot=True, fmt='g', cmap='Blues')
plt.xlabel('Overall rating')
plt.ylabel('Speaker clarity rating')
plt.title('Speaker clarity by overall rating: respondent counts')
plt.tight_layout()
plt.savefig(figures / 'crispdm_speaker_clarity_overall_heatmap.png', dpi=180)
plt.show()

## CRISP-DM 6 — Deployment and decision handoff

Deployment here means producing traceable tables, figures, a manifest, and a decision-oriented readout for Chapter 3 and Chapter 4. The generated prompts below are evidence pointers for human interpretation; they are not automatically written conclusions. Re-run the notebook after any approved data, coding, or pipeline change.

In [ ]:
top_item = item_plot.iloc[-1]
lowest_item = item_plot.iloc[0]
substantive_theme_counts = theme_counts.loc[theme_counts['code_id'].ne('NO_SUBSTANTIVE_RESPONSE')].copy()
top_theme = substantive_theme_counts.sort_values(['n_assignments', 'code_id'], ascending=[False, True]).iloc[0] if len(substantive_theme_counts) else None
decision_readout = {
    'sample_n': int(len(raw)),
    'flagged_scale_direction_rows': int(manifest['flagged_rows']),
    'strongest_primary_item': {'item_key': top_item['item_key'], 'label': top_item['item_label'], 'mean': float(top_item['mean']), 'agree_pct': float(top_item['agree_pct_4_or_5'])},
    'lowest_primary_item': {'item_key': lowest_item['item_key'], 'label': lowest_item['item_label'], 'mean': float(lowest_item['mean']), 'agree_pct': float(lowest_item['agree_pct_4_or_5'])},
    'qualitative_priority_code': None if top_theme is None else {'code_id': top_theme['code_id'], 'label': top_theme['label'], 'assignments': int(top_theme['n_assignments']), 'respondents': int(top_theme['n_respondents'])},
    'interpretation_boundary': 'Descriptive post-event perceptions and coded reported feedback; not objective learning, causal impact, or population estimates.',
}
(tables / 'crispdm_decision_readout.json').write_text(json.dumps(decision_readout, indent=2), encoding='utf-8')
decision_prompts = pd.DataFrame([
    {'decision_area': 'Retain or showcase strength', 'evidence': f"{top_item['item_label']} (mean {top_item['mean']:.2f}; agreement {top_item['agree_pct_4_or_5']:.2f}%)"},
    {'decision_area': 'Investigate improvement opportunity', 'evidence': f"{lowest_item['item_label']} (mean {lowest_item['mean']:.2f}; agreement {lowest_item['agree_pct_4_or_5']:.2f}%)"},
    {'decision_area': 'Triangulate with participant voice', 'evidence': 'Compare the lowest-rated dimensions with thematic codes and prompt-specific response coverage before recommending changes.'},
    {'decision_area': 'Protect data quality', 'evidence': f"Review the {int(manifest['flagged_rows'])} scale-direction flag(s) and preserve the normalization audit with every manuscript result."},
])
display(decision_prompts)
decision_prompts.to_csv(tables / 'crispdm_decision_prompts.csv', index=False)
print('CRISP-DM outputs written to:', output_dir)

## Copy-ready quantitative and mixed-methods findings

This cell creates a compact reporting layer using the verified tables. It is intended as a drafting aid for Results and Discussion, not as a substitute for checking the denominators, intervals, assumptions, and qualitative evidence matrix.

In [ ]:
def summary_row(key):
    return summary.loc[summary['item_key'].eq(key)].iloc[0]

overall_row = summary_row('overall_rating')
satisfaction_row = summary_row('satisfaction')
strong_clarity = speaker_summary.loc[speaker_summary[clarity_key].eq(5)]
strong_clarity_text = 'not estimable' if strong_clarity.empty else f"{float(strong_clarity.iloc[0]['top_box_pct']):.2f}% ({int(strong_clarity.iloc[0]['top_box_n'])}/{int(strong_clarity.iloc[0]['n'])})"

report_lines = [
    '### Key performance highlights',
    f"- The analytic sample contained {len(raw)} submitted responses. The 13-item Likert matrix was summarized after the pre-specified normalization rule flagged {int(manifest['flagged_rows'])} respondent row(s).",
    f"- Overall satisfaction: {int(satisfaction_row['count_5'])}/{int(satisfaction_row['n'])} respondents ({float(satisfaction_row['pct_5']):.2f}%) selected 5/5; {float(satisfaction_row['agree_pct']):.2f}% selected 4 or 5.",
    f"- Overall session rating: {int(overall_row['count_5'])}/{int(overall_row['n'])} respondents ({float(overall_row['pct_5']):.2f}%) selected 5/5; {float(overall_row['agree_pct']):.2f}% selected 4 or 5.",
    f"- Highest primary dimension: {top_item['item_label']} (mean {float(top_item['mean']):.2f}; agreement {float(top_item['agree_pct_4_or_5']):.2f}%; top-box {float(top_item['top_box_pct_5']):.2f}%).",
    f"- Lowest primary dimension: {lowest_item['item_label']} (mean {float(lowest_item['mean']):.2f}; agreement {float(lowest_item['agree_pct_4_or_5']):.2f}%; top-box {float(lowest_item['top_box_pct_5']):.2f}%).",
    f"- Among respondents selecting the strongest speaker-clarity category, the overall-rating 5/5 rate was {strong_clarity_text}; this is a descriptive cross-tab, not evidence of a causal effect.",
    f"- The ordinal association between speaker clarity and overall rating was Spearman rho = {float(association_summary.iloc[0]['spearman_rho']):.4f} across {int(association_summary.iloc[0]['n'])} complete cases; interpret this as descriptive, not causal.",
    '',
    '### Qualitative evidence and action cues',
    '- Theme frequencies are coded assignments and may include multiple codes per respondent answer; they are not counts of objectively learned outcomes.',
    f"- The institutional distribution and prompt-level response coverage are available in `crispdm_institution_summary.csv` and `crispdm_data_profile.csv`.",
    f"- The mixed-methods evidence matrix is available in `mixed_methods_evidence_matrix.csv`; use it to compare rating strengths and weaknesses with reported learning, valued features, and improvement requests.",
    '',
    '### Interpretation boundary',
    '- These results describe post-event perceptions from this respondent set. They do not establish objective learning, causal impact, psychometric validity, or population-level generalizability.',
]
report_text = '\n'.join(report_lines)
print(report_text)
(tables / 'crispdm_copy_ready_findings.md').write_text(report_text + '\n', encoding='utf-8')

In [ ]:
import pandas as pd

tables = output_dir / 'tables'
likert = pd.read_csv(tables / 'likert_item_summary.csv')
summary = pd.read_csv(tables / 'overall_satisfaction_summary.csv')
themes = pd.read_csv(tables / 'thematic_counts.csv')
display(likert[['item_order', 'item_label', 'n', 'mean', 'sd', 'agree_pct']])
display(summary[['item_key', 'n', 'mean', 'sd', 'agree_pct']])
display(themes)

In [ ]:
from IPython.display import Image, display

figures = output_dir / 'figures'
display(Image(filename=str(figures / 'likert_item_means.png')))
display(Image(filename=str(figures / 'likert_distribution.png')))

In [ ]:
matrix = pd.read_csv(tables / 'mixed_methods_evidence_matrix.csv')
display(matrix)
print('Copy-ready quantitative table:')
print(likert[['item_order', 'item_label', 'n', 'mean', 'sd', 'agree_pct']].to_markdown(index=False))
print('Copy-ready summary table:')
print(summary[['item_key', 'n', 'mean', 'sd', 'agree_pct']].to_markdown(index=False))
print('Copy-ready thematic counts:')
print(themes.to_markdown(index=False))

## Interpretation boundary

The normalized Likert values describe respondents' post-event perceptions. Overall rating and satisfaction are retained as separate single-item summaries. The two flagged rows are corrected only for the 13 primary Likert items because their all-1 item pattern conflicts with their positive summary ratings. Open-ended code frequencies describe reported feedback and are not objective measures of learning. The design is nonexperimental and post-event, so these outputs do not establish causation, objective competence, or population-level generalizability.

## Figure gallery

This gallery displays every PNG generated or retained in `ANALYSIS/outputs/figures/`. The files are written by the reproducible pipeline and are the figures linked in Chapter 3.

In [ ]:
from IPython.display import Image, Markdown, display

figure_paths = sorted((output_dir / 'figures').glob('*.png'))
print(f'{len(figure_paths)} figure files available')
for figure_path in figure_paths:
    display(Markdown(f'### `{figure_path.name}`'))
    display(Image(filename=str(figure_path), width=900))

### Static figure links

![Agreement and top-box](../outputs/figures/agreement_topbox_scatter.png)

![Composite distribution](../outputs/figures/composite_distribution.png)

![Item means with bootstrap intervals](../outputs/figures/likert_item_means.png)

![Item means with interquartile ranges](../outputs/figures/likert_item_means_iqr.png)

![Likert distribution](../outputs/figures/likert_distribution.png)

![Summary rating distribution](../outputs/figures/summary_rating_distribution.png)

![Institution profile](../outputs/figures/institution_profile.png)

![Speaker clarity and overall rating](../outputs/figures/speaker_clarity_overall_heatmap.png)

![Theme assignments](../outputs/figures/theme_assignments.png)

![Theme by prompt](../outputs/figures/theme_by_prompt_heatmap.png)

![Normalization sensitivity](../outputs/figures/normalization_sensitivity.png)

![CRISP-DM item means](../outputs/figures/crispdm_item_means_with_ci.png)

![CRISP-DM speaker heatmap](../outputs/figures/crispdm_speaker_clarity_overall_heatmap.png)

# Appendix 1 — Relevant source code

The following displays contain the complete Python source used for data loading, validation, Likert normalization, summary statistics, mixed-methods tables, and figure generation. The notebook imports these same files during the CRISP-DM run; the repository modules remain the authoritative executable source, while this appendix makes the relevant code visible in the submitted notebook.

In [ ]:
from IPython.display import Code, Markdown, display

appendix_source_files = [
    PROJECT_ROOT / 'ANALYSIS' / 'src' / 'pipeline' / 'config.py',
    PROJECT_ROOT / 'ANALYSIS' / 'src' / 'pipeline' / 'io.py',
    PROJECT_ROOT / 'ANALYSIS' / 'src' / 'pipeline' / 'quality.py',
    PROJECT_ROOT / 'ANALYSIS' / 'src' / 'pipeline' / 'summary.py',
    PROJECT_ROOT / 'ANALYSIS' / 'src' / 'pipeline' / 'reporting.py',
    PROJECT_ROOT / 'ANALYSIS' / 'src' / 'pipeline' / 'run_analysis.py',
]
for source_path in appendix_source_files:
    relative_path = source_path.relative_to(PROJECT_ROOT)
    display(Markdown(f'## `{relative_path.as_posix()}`'))
    display(Code(filename=str(source_path), language='python'))

### `ANALYSIS/src/pipeline/config.py`

```python
from __future__ import annotations

from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parents[3]
ANALYSIS_ROOT = REPO_ROOT / "ANALYSIS"
DEFAULT_INPUT = ANALYSIS_ROOT / "data" / "CodeCrafters_ Tech Beyond the Screen – Day 1 Afternoon Session Evaluation.csv"
DEFAULT_OUTPUT = ANALYSIS_ROOT / "outputs"

# These indices are stable for the supplied export: timestamp, three consent
# fields, institution, 13 experience items, overall rating, satisfaction, and
# four open-ended fields. Headers are intentionally selected by position so
# trailing spaces and spreadsheet line breaks cannot change the mapping.
PRIMARY_ITEM_KEYS = [f"item_{i:02d}" for i in range(1, 14)]
PRIMARY_ITEM_LABELS = [
    "Session objectives were clearly presented",
    "Topics were relevant and informative",
    "Content was easy to understand",
    "Speakers explained the topics clearly",
    "Presentation materials were clear and organized",
    "Examples and discussions supported understanding",
    "Session was engaging and interactive",
    "Activities reinforced the topics",
    "Knowledge was useful for studies or everyday life",
    "Session encouraged critical thinking",
    "Session was well-organized",
    "Technical aspects were satisfactory",
    "Session met expectations",
]
PRIMARY_COLUMN_INDICES = list(range(5, 18))
SUMMARY_KEYS = ["overall_rating", "satisfaction"]
SUMMARY_COLUMN_INDICES = [18, 19]
TEXT_KEYS = ["learning_value", "liked_most", "improvements", "additional_comments"]
TEXT_COLUMN_INDICES = [20, 21, 22, 23]
ALL_RATING_KEYS = PRIMARY_ITEM_KEYS + SUMMARY_KEYS

THEME_CODEBOOK_PATH = ANALYSIS_ROOT / "src" / "themes" / "codebook.json"
THEME_DECISIONS_PATH = ANALYSIS_ROOT / "src" / "themes" / "coding_decisions.csv"


```

### `ANALYSIS/src/pipeline/io.py`

```python
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

from .config import (
    PRIMARY_COLUMN_INDICES,
    PRIMARY_ITEM_KEYS,
    SUMMARY_COLUMN_INDICES,
    SUMMARY_KEYS,
    TEXT_COLUMN_INDICES,
    TEXT_KEYS,
)


def load_evaluation_csv(path: Path) -> pd.DataFrame:
    """Load the raw export without modifying or overwriting it."""
    frame = pd.read_csv(path, encoding="utf-8")
    frame.insert(0, "respondent_id", [f"R{i:03d}" for i in range(1, len(frame) + 1)])
    return frame


def extract_rating(value: object) -> int | pd.NA:
    """Parse the first valid 1–5 rating from a spreadsheet cell."""
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (int, float)) and float(value).is_integer() and 1 <= int(value) <= 5:
        return int(value)
    match = re.search(r"(?<!\d)([1-5])(?!\d)", str(value))
    return int(match.group(1)) if match else pd.NA


def _parsed_series(frame: pd.DataFrame, source_index: int) -> pd.Series:
    return frame.iloc[:, source_index + 1].map(extract_rating).astype("Int64")


def build_rating_frame(raw: pd.DataFrame) -> pd.DataFrame:
    """Create stable analysis names for the 13 items and two summaries."""
    result = pd.DataFrame({"respondent_id": raw["respondent_id"]})
    for key, source_index in zip(PRIMARY_ITEM_KEYS, PRIMARY_COLUMN_INDICES):
        result[key] = _parsed_series(raw, source_index)
    for key, source_index in zip(SUMMARY_KEYS, SUMMARY_COLUMN_INDICES):
        result[key] = _parsed_series(raw, source_index)
    return result


def build_text_frame(raw: pd.DataFrame) -> pd.DataFrame:
    """Extract open-ended fields with stable names and de-identified row IDs."""
    result = pd.DataFrame({"respondent_id": raw["respondent_id"]})
    for key, source_index in zip(TEXT_KEYS, TEXT_COLUMN_INDICES):
        result[key] = raw.iloc[:, source_index + 1].fillna("").astype(str).str.strip()
    return result


```

### `ANALYSIS/src/pipeline/quality.py`

```python
from __future__ import annotations

from typing import Iterable

import pandas as pd

from .config import PRIMARY_ITEM_KEYS, SUMMARY_KEYS


def validate_frame(raw: pd.DataFrame, ratings: pd.DataFrame) -> dict:
    required = set(["respondent_id", *PRIMARY_ITEM_KEYS, *SUMMARY_KEYS])
    missing_columns = sorted(required.difference(ratings.columns))
    rating_values = ratings[PRIMARY_ITEM_KEYS + SUMMARY_KEYS]
    invalid_cells = int(rating_values.isna().sum().sum())
    out_of_range = int(((rating_values < 1) | (rating_values > 5)).fillna(False).sum().sum())
    return {
        "rows": int(len(raw)),
        "raw_columns": int(len(raw.columns) - 1),
        "missing_analysis_columns": missing_columns,
        "invalid_rating_cells": invalid_cells,
        "out_of_range_cells": out_of_range,
    }


def flag_scale_direction_rows(ratings: pd.DataFrame) -> pd.DataFrame:
    primary = ratings[PRIMARY_ITEM_KEYS]
    summaries = ratings[SUMMARY_KEYS]
    all_primary_one = primary.eq(1).all(axis=1)
    summaries_high = summaries.ge(4).all(axis=1)
    flagged = all_primary_one & summaries_high
    return pd.DataFrame(
        {
            "respondent_id": ratings["respondent_id"],
            "all_primary_items_raw_one": all_primary_one,
            "summary_ratings_at_least_four": summaries_high,
            "flagged": flagged,
            "reason": flagged.map(
                lambda value: "Primary Likert scale direction appears reversed; summary ratings are positive"
                if value
                else ""
            ),
        }
    )


def normalize_rating_frame(
    ratings: pd.DataFrame, flagged_ids: Iterable[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Reverse only primary Likert items; retain summary anchors unchanged."""
    normalized = ratings.copy()
    flagged_set = set(flagged_ids)
    audit_rows: list[dict] = []
    for respondent_id in sorted(flagged_set):
        mask = normalized["respondent_id"].eq(respondent_id)
        for key in PRIMARY_ITEM_KEYS:
            raw_value = normalized.loc[mask, key].iloc[0]
            new_value = 6 - int(raw_value)
            normalized.loc[mask, key] = new_value
            audit_rows.append(
                {
                    "respondent_id": respondent_id,
                    "field": key,
                    "raw_value": int(raw_value),
                    "normalized_value": new_value,
                    "reason": "Approved respondent-level Likert direction correction",
                }
            )
    return normalized, pd.DataFrame(audit_rows)


```

### `ANALYSIS/src/pipeline/summary.py`

```python
from __future__ import annotations

import pandas as pd


def _distribution(values: pd.Series) -> dict:
    counts = values.value_counts().reindex(range(1, 6), fill_value=0)
    n = int(values.notna().sum())
    return {
        **{f"count_{score}": int(counts[score]) for score in range(1, 6)},
        **{f"pct_{score}": round(float(counts[score] / n * 100), 2) if n else 0.0 for score in range(1, 6)},
    }


def summarize_likert(frame: pd.DataFrame, columns: list[str], labels: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    distributions = []
    for order, (column, label) in enumerate(zip(columns, labels), start=1):
        values = frame[column].astype("Float64")
        nonmissing = values.dropna()
        row = {
            "item_order": order,
            "item_key": column,
            "item_label": label,
            "n": int(nonmissing.size),
            "mean": round(float(nonmissing.mean()), 4),
            "median": float(nonmissing.median()),
            "sd": round(float(nonmissing.std(ddof=1)), 4),
            "min": int(nonmissing.min()),
            "max": int(nonmissing.max()),
            "q1": float(nonmissing.quantile(0.25)),
            "q3": float(nonmissing.quantile(0.75)),
            "agree_n": int((nonmissing >= 4).sum()),
            "agree_pct": round(float((nonmissing >= 4).mean() * 100), 2),
        }
        rows.append(row)
        distributions.append({"item_key": column, "item_label": label, **_distribution(values)})
    return pd.DataFrame(rows), pd.DataFrame(distributions)


def summarize_single_items(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for column in columns:
        values = frame[column].astype("Float64").dropna()
        rows.append(
            {
                "item_key": column,
                "n": int(values.size),
                "mean": round(float(values.mean()), 4),
                "median": float(values.median()),
                "sd": round(float(values.std(ddof=1)), 4),
                "min": int(values.min()),
                "max": int(values.max()),
                "agree_n": int((values >= 4).sum()),
                "agree_pct": round(float((values >= 4).mean() * 100), 2),
                **_distribution(values),
            }
        )
    return pd.DataFrame(rows)


```

### `ANALYSIS/src/pipeline/reporting.py`

```python
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from .config import PRIMARY_ITEM_KEYS, PRIMARY_ITEM_LABELS, SUMMARY_KEYS, TEXT_KEYS


BOOTSTRAP_SEED = 20261012
BOOTSTRAP_REPS = 2000


def _wilson_interval(successes: int, total: int, z: float = 1.96) -> tuple[float, float]:
    if total == 0:
        return 0.0, 0.0
    p = successes / total
    denominator = 1 + z**2 / total
    centre = (p + z**2 / (2 * total)) / denominator
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * total)) / total) / denominator
    return (max(0.0, centre - margin) * 100, min(1.0, centre + margin) * 100)


def _bootstrap_mean_interval(values: pd.Series, rng: np.random.Generator) -> tuple[float, float]:
    numeric = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if numeric.size < 2:
        mean = float(numeric.mean()) if numeric.size else np.nan
        return mean, mean
    sample = rng.choice(numeric, size=(BOOTSTRAP_REPS, numeric.size), replace=True).mean(axis=1)
    return tuple(np.quantile(sample, [0.025, 0.975]).tolist())


def _metric_rows(
    normalized: pd.DataFrame,
    columns: list[str],
    labels: list[str],
    source: str,
) -> pd.DataFrame:
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    rows: list[dict] = []
    for key, label in zip(columns, labels):
        values = pd.to_numeric(normalized[key], errors="coerce").dropna()
        n = int(values.size)
        agree_n = int((values >= 4).sum())
        top_box_n = int((values == 5).sum())
        mean_low, mean_high = _bootstrap_mean_interval(values, rng)
        agree_low, agree_high = _wilson_interval(agree_n, n)
        rows.append(
            {
                "source": source,
                "item_key": key,
                "item_label": label,
                "n": n,
                "mean": round(float(values.mean()), 4),
                "median": float(values.median()),
                "sd": round(float(values.std(ddof=1)), 4),
                "q1": round(float(values.quantile(0.25)), 4),
                "q3": round(float(values.quantile(0.75)), 4),
                "min": int(values.min()),
                "max": int(values.max()),
                "agree_n": agree_n,
                "agree_pct": round(agree_n / n * 100, 2),
                "agree_ci_low": round(agree_low, 2),
                "agree_ci_high": round(agree_high, 2),
                "top_box_n": top_box_n,
                "top_box_pct": round(top_box_n / n * 100, 2),
                "mean_ci_low": round(float(mean_low), 4),
                "mean_ci_high": round(float(mean_high), 4),
            }
        )
    return pd.DataFrame(rows)


def _composite_summary(normalized: pd.DataFrame) -> pd.DataFrame:
    composite = normalized[PRIMARY_ITEM_KEYS].astype(float).mean(axis=1)
    return pd.DataFrame(
        [
            {
                "metric": "13-item normalized experience composite",
                "n": int(composite.notna().sum()),
                "mean": round(float(composite.mean()), 4),
                "median": round(float(composite.median()), 4),
                "sd": round(float(composite.std(ddof=1)), 4),
                "min": round(float(composite.min()), 4),
                "max": round(float(composite.max()), 4),
                "agree_composite_pct": round(float((composite >= 4).mean() * 100), 2),
            }
        ]
    )


def _institution_summary(raw: pd.DataFrame) -> pd.DataFrame:
    # The institution field is the fifth source column; respondent_id was inserted at 0.
    institution = raw.iloc[:, 5].fillna("Unspecified/NA").astype(str).str.strip()
    institution = institution.replace({"": "Unspecified/NA", "nan": "Unspecified/NA"})
    counts = institution.value_counts(dropna=False).rename_axis("institution").reset_index(name="n")
    counts["pct"] = (counts["n"] / len(institution) * 100).round(2)
    return counts


def _speaker_tables(normalized: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    cross = pd.crosstab(normalized["item_04"], normalized["overall_rating"]).reindex(
        index=range(1, 6), columns=range(1, 6), fill_value=0
    )
    clarity_labels = {1: "1 - Strongly disagree", 2: "2", 3: "3", 4: "4", 5: "5 - Strongly agree"}
    cross.index = [clarity_labels[i] for i in cross.index]
    cross.index.name = None
    summary = (
        normalized.groupby("item_04", dropna=False)
        .agg(
            n=("overall_rating", "size"),
            mean_overall=("overall_rating", "mean"),
            top_box_n=("overall_rating", lambda values: int((values == 5).sum())),
        )
        .reset_index()
    )
    summary["top_box_pct"] = (summary["top_box_n"] / summary["n"] * 100).round(2)
    summary["clarity_label"] = summary["item_04"].map(clarity_labels)
    rho = normalized[["item_04", "overall_rating"]].corr(method="spearman").iloc[0, 1]
    association = pd.DataFrame(
        [
            {
                "comparison": "speaker clarity vs overall rating",
                "n": int(len(normalized)),
                "spearman_rho": round(float(rho), 4),
                "interpretation": "Descriptive ordinal association; not causal.",
            }
        ]
    )
    return cross, summary, association


def _theme_tables(theme_counts: pd.DataFrame, decisions: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    if theme_counts.empty or decisions.empty:
        empty = pd.DataFrame(columns=["source_field", "code_id", "label", "n_assignments"])
        return theme_counts.copy(), {"wide": empty, "long": empty}
    extended = theme_counts.copy()
    respondents = decisions["respondent_id"].nunique()
    assignments = len(decisions)
    extended["respondent_coverage_pct"] = (extended["n_respondents"] / respondents * 100).round(2)
    extended["assignment_share_pct"] = (extended["n_assignments"] / assignments * 100).round(2)
    by_prompt = pd.crosstab(decisions["source_field"], decisions["code_id"])
    by_prompt = by_prompt.reindex(sorted(by_prompt.columns), axis=1)
    labels = extended[["code_id", "label"]].drop_duplicates()
    long = decisions.groupby(["source_field", "code_id"], as_index=False).size().rename(columns={"size": "n_assignments"})
    long = long.merge(labels, on="code_id", how="left")
    return extended, {"wide": by_prompt, "long": long[["source_field", "code_id", "label", "n_assignments"]]}


def build_extended_tables(
    raw: pd.DataFrame,
    ratings: pd.DataFrame,
    normalized: pd.DataFrame,
    item_summary: pd.DataFrame,
    distributions: pd.DataFrame,
    single_summary: pd.DataFrame,
    theme_counts: pd.DataFrame,
    theme_decisions: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    del item_summary, distributions, single_summary
    labels = PRIMARY_ITEM_LABELS + ["Overall Rating", "Satisfaction"]
    metrics = _metric_rows(normalized, PRIMARY_ITEM_KEYS + SUMMARY_KEYS, labels, "normalized ratings")
    composite = _composite_summary(normalized)
    institutions = _institution_summary(raw)
    speaker_cross, speaker_summary, association = _speaker_tables(normalized)
    themes, theme_by_prompt = _theme_tables(theme_counts, theme_decisions)
    raw_means = ratings[PRIMARY_ITEM_KEYS].astype(float).mean()
    normalized_means = normalized[PRIMARY_ITEM_KEYS].astype(float).mean()
    sensitivity = pd.DataFrame(
        {
            "item_key": PRIMARY_ITEM_KEYS,
            "item_label": PRIMARY_ITEM_LABELS,
            "raw_mean": [raw_means[key] for key in PRIMARY_ITEM_KEYS],
            "normalized_mean": [normalized_means[key] for key in PRIMARY_ITEM_KEYS],
        }
    )
    sensitivity["mean_change"] = sensitivity["normalized_mean"] - sensitivity["raw_mean"]
    sensitivity = sensitivity.round(4)
    return {
        "quantitative_metrics": metrics,
        "composite_summary": composite,
        "institution_summary": institutions,
        "speaker_cross": speaker_cross,
        "speaker_summary": speaker_summary,
        "association_summary": association,
        "theme_counts": themes,
        "theme_by_prompt": theme_by_prompt["wide"],
        "theme_by_prompt_long": theme_by_prompt["long"],
        "normalization_sensitivity": sensitivity,
        "composite_values": pd.DataFrame({"composite": normalized[PRIMARY_ITEM_KEYS].astype(float).mean(axis=1)}),
    }


def write_extended_tables(tables_dir: Path, tables: dict[str, pd.DataFrame]) -> None:
    filenames = {
        "quantitative_metrics": "crispdm_quantitative_metrics.csv",
        "composite_summary": "crispdm_composite_summary.csv",
        "institution_summary": "crispdm_institution_summary.csv",
        "speaker_cross": "crispdm_speaker_clarity_overall_crosstab.csv",
        "speaker_summary": "crispdm_speaker_clarity_overall_summary.csv",
        "association_summary": "crispdm_association_summary.csv",
        "theme_counts": "crispdm_theme_counts_with_denominators.csv",
        "theme_by_prompt": "crispdm_theme_by_prompt.csv",
        "normalization_sensitivity": "crispdm_normalization_sensitivity.csv",
    }
    for key, filename in filenames.items():
        if key == "quantitative_metrics":
            frame = tables[key].rename(
                columns={
                    "source": "stream",
                    "agree_n": "agree_n_4_or_5",
                    "agree_pct": "agree_pct_4_or_5",
                    "agree_ci_low": "agree_ci95_low",
                    "agree_ci_high": "agree_ci95_high",
                    "top_box_n": "top_box_n_5",
                    "top_box_pct": "top_box_pct_5",
                    "mean_ci_low": "mean_bootstrap_ci95_low",
                    "mean_ci_high": "mean_bootstrap_ci95_high",
                }
            ).copy()
            frame["stream"] = frame["stream"].replace({"normalized ratings": "primary_likert"})
            frame.loc[frame["item_key"].isin(SUMMARY_KEYS), "stream"] = "summary_rating"
            frame = frame.drop(columns=["q1", "q3"], errors="ignore")
            frame = frame[["stream", "item_key", "item_label", "n", "mean", "median", "sd", "min", "max", "agree_n_4_or_5", "agree_pct_4_or_5", "agree_ci95_low", "agree_ci95_high", "top_box_n_5", "top_box_pct_5", "mean_bootstrap_ci95_low", "mean_bootstrap_ci95_high"]]
        elif key == "theme_by_prompt":
            frame = tables[key]
            frame.to_csv(tables_dir / filename)
            continue
        elif key == "speaker_cross":
            frame = tables[key]
            frame.to_csv(tables_dir / filename)
            continue
        elif key == "normalization_sensitivity":
            frame = tables[key][["item_key", "raw_mean", "normalized_mean", "mean_change"]]
        else:
            frame = tables[key]
        frame.to_csv(tables_dir / filename, index=False)


def _save(fig: plt.Figure, figure_dir: Path, filename: str) -> None:
    fig.tight_layout()
    fig.savefig(figure_dir / filename, dpi=180, bbox_inches="tight")
    plt.close(fig)


def write_extended_figures(tables: dict[str, pd.DataFrame], figure_dir: Path) -> None:
    sns.set_theme(style="whitegrid", context="notebook")
    metrics = tables["quantitative_metrics"]
    primary = metrics[metrics["item_key"].isin(PRIMARY_ITEM_KEYS)].copy()
    primary = primary.sort_values("mean")

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.errorbar(primary["mean"], primary["item_label"], xerr=[primary["mean"] - primary["mean_ci_low"], primary["mean_ci_high"] - primary["mean"]], fmt="o", color="#2f6690", ecolor="#8fb8d4", capsize=3)
    ax.set_xlim(1, 5)
    ax.set_xlabel("Mean normalized rating (1-5), with 95% bootstrap interval")
    ax.set_ylabel("")
    _save(fig, figure_dir, "likert_item_means.png")

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.hlines(primary["item_label"], primary["q1"] if "q1" in primary else primary["mean"], primary["q3"] if "q3" in primary else primary["mean"], color="#8fb8d4")
    ax.scatter(primary["mean"], primary["item_label"], color="#2f6690", zorder=3)
    ax.set_xlim(1, 5)
    ax.set_xlabel("Mean normalized rating (1-5); point with interquartile range")
    ax.set_ylabel("")
    _save(fig, figure_dir, "likert_item_means_iqr.png")

    dist = tables["distributions"].copy() if "distributions" in tables else None
    if dist is not None:
        long = dist.melt(id_vars=["item_key", "item_label"], value_vars=[f"pct_{i}" for i in range(1, 6)], var_name="rating", value_name="percent")
        long["rating"] = long["rating"].str.replace("pct_", "", regex=False).astype(int)
        pivot = long.pivot(index="item_label", columns="rating", values="percent").reindex(primary["item_label"])
        fig, ax = plt.subplots(figsize=(11, 8))
        pivot.plot.barh(stacked=True, ax=ax, colormap="viridis", width=0.8)
        ax.set_xlabel("Responses (%)")
        ax.set_ylabel("")
        ax.legend(title="Rating", ncol=5, loc="lower right")
        _save(fig, figure_dir, "likert_distribution.png")

    summary = metrics[metrics["item_key"].isin(SUMMARY_KEYS)].copy()
    summary["item_label"] = summary["item_key"].map({"overall_rating": "Overall session rating", "satisfaction": "Overall satisfaction"})
    fig, ax = plt.subplots(figsize=(9, 4))
    summary_plot = summary.set_index("item_label")[[f"{col}_{score}" for col in ("count",) for score in range(1, 6)]] if all(f"count_{score}" in summary for score in range(1, 6)) else None
    if summary_plot is None:
        # Counts are derived from the corresponding normalized rows for the two summary items.
        values = tables["summary_values"]
        summary_plot = values.groupby("item_label")["rating"].value_counts().unstack(fill_value=0).reindex(columns=range(1, 6), fill_value=0)
    summary_plot.div(summary_plot.sum(axis=1), axis=0).mul(100).plot.barh(stacked=True, ax=ax, colormap="viridis")
    ax.set_xlabel("Responses (%)")
    ax.set_ylabel("")
    ax.legend(title="Rating", ncol=5, loc="lower right")
    _save(fig, figure_dir, "summary_rating_distribution.png")

    composite = tables["composite_values"]["composite"].dropna()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(composite, bins=np.arange(2.75, 5.26, 0.25), color="#2f6690", edgecolor="white")
    ax.axvline(composite.mean(), color="#d1495b", linestyle="--", label=f"Mean = {composite.mean():.2f}")
    ax.axvline(composite.median(), color="#edae49", linestyle=":", label=f"Median = {composite.median():.2f}")
    ax.set_xlabel("Respondent-level normalized 13-item mean")
    ax.set_ylabel("Respondents")
    ax.legend()
    _save(fig, figure_dir, "composite_distribution.png")

    cross = tables["speaker_cross"].copy()
    if "speaker_clarity" in cross.columns:
        cross = cross.set_index("speaker_clarity")
    cross = cross.reindex(columns=range(1, 6), fill_value=0)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cross, annot=True, fmt="g", cmap="Blues", cbar=False, ax=ax)
    ax.set_xlabel("Overall session rating")
    ax.set_ylabel("Speaker clarity rating")
    _save(fig, figure_dir, "speaker_clarity_overall_heatmap.png")

    institutions = tables["institution_summary"].sort_values("n")
    fig, ax = plt.subplots(figsize=(10, max(5, len(institutions) * 0.35)))
    ax.barh(institutions["institution"], institutions["n"], color="#4c956c")
    ax.set_xlabel("Respondents")
    ax.set_ylabel("")
    _save(fig, figure_dir, "institution_profile.png")

    themes = tables["theme_counts"]
    themes = themes[themes["code_id"] != "NO_SUBSTANTIVE_RESPONSE"].sort_values("n_assignments")
    if not themes.empty:
        fig, ax = plt.subplots(figsize=(10, max(5, len(themes) * 0.35)))
        ax.barh(themes["label"], themes["n_assignments"], color="#6c5b7b")
        ax.set_xlabel("Coded assignments")
        ax.set_ylabel("")
        _save(fig, figure_dir, "theme_assignments.png")

        prompt = tables["theme_by_prompt_long"]
        prompt = prompt[prompt["code_id"] != "NO_SUBSTANTIVE_RESPONSE"]
        matrix = prompt.pivot_table(index="label", columns="source_field", values="n_assignments", fill_value=0)
        fig, ax = plt.subplots(figsize=(9, max(5, len(matrix) * 0.35)))
        sns.heatmap(matrix, annot=True, fmt="g", cmap="Purples", cbar=False, ax=ax)
        ax.set_xlabel("Open-ended prompt")
        ax.set_ylabel("")
        _save(fig, figure_dir, "theme_by_prompt_heatmap.png")

    sensitivity = tables["normalization_sensitivity"].sort_values("mean_change")
    fig, ax = plt.subplots(figsize=(10, 7))
    y = np.arange(len(sensitivity))
    ax.scatter(sensitivity["raw_mean"], y, label="Raw mean", color="#999999")
    ax.scatter(sensitivity["normalized_mean"], y, label="Normalized mean", color="#2f6690")
    for raw_mean, normalized_mean, yy in zip(sensitivity["raw_mean"], sensitivity["normalized_mean"], y):
        ax.plot([raw_mean, normalized_mean], [yy, yy], color="#d1495b", linewidth=1)
    ax.set_yticks(y, sensitivity["item_label"])
    ax.set_xlim(1, 5)
    ax.set_xlabel("Mean rating (1-5)")
    ax.set_ylabel("")
    ax.legend()
    _save(fig, figure_dir, "normalization_sensitivity.png")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(primary["agree_pct"], primary["top_box_pct"], color="#2f6690")
    for row in primary.itertuples(index=False):
        ax.annotate(row.item_key.replace("item_", ""), (row.agree_pct, row.top_box_pct), xytext=(4, 3), textcoords="offset points", fontsize=8)
    ax.set_xlabel("Agreement (ratings 4-5), %")
    ax.set_ylabel("Top-box (rating 5), %")
    ax.set_xlim(85, 101)
    ax.set_ylim(0, 101)
    _save(fig, figure_dir, "agreement_topbox_scatter.png")


def add_derived_plot_inputs(
    tables: dict[str, pd.DataFrame], normalized: pd.DataFrame, distributions: pd.DataFrame
) -> dict[str, pd.DataFrame]:
    tables = dict(tables)
    tables["distributions"] = distributions
    tables["summary_values"] = pd.concat(
        [
            normalized[["overall_rating"]].rename(columns={"overall_rating": "rating"}).assign(item_label="Overall session rating"),
            normalized[["satisfaction"]].rename(columns={"satisfaction": "rating"}).assign(item_label="Overall satisfaction"),
        ],
        ignore_index=True,
    )
    return tables


```

### `ANALYSIS/src/pipeline/run_analysis.py`

```python
from __future__ import annotations

import argparse
import hashlib
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from .config import (
    DEFAULT_INPUT,
    DEFAULT_OUTPUT,
    PRIMARY_ITEM_KEYS,
    PRIMARY_ITEM_LABELS,
    SUMMARY_KEYS,
    THEME_CODEBOOK_PATH,
    THEME_DECISIONS_PATH,
)
from .io import build_rating_frame, build_text_frame, load_evaluation_csv
from .quality import flag_scale_direction_rows, normalize_rating_frame, validate_frame
from .reporting import (
    add_derived_plot_inputs,
    build_extended_tables,
    write_extended_figures,
    write_extended_tables,
)
from .summary import summarize_likert, summarize_single_items


def _ensure_dirs(output_dir: Path) -> dict[str, Path]:
    paths = {
        "tables": output_dir / "tables",
        "processed": output_dir / "processed",
        "figures": output_dir / "figures",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def _theme_outputs(output_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    counts_path = output_dir / "tables" / "thematic_counts.csv"
    matrix_path = output_dir / "tables" / "mixed_methods_evidence_matrix.csv"
    if not THEME_DECISIONS_PATH.exists():
        empty = pd.DataFrame(columns=["code_id", "label", "n_assignments", "n_respondents"])
        empty.to_csv(counts_path, index=False)
        pd.DataFrame(columns=["evidence_stream", "measure", "value", "interpretive_boundary"]).to_csv(matrix_path, index=False)
        return empty, pd.DataFrame()

    decisions = pd.read_csv(THEME_DECISIONS_PATH)
    required = {"respondent_id", "source_field", "code_id", "coding_note"}
    missing = sorted(required.difference(decisions.columns))
    if missing:
        raise ValueError(f"Coding decisions are missing columns: {missing}")
    codebook = json.loads(THEME_CODEBOOK_PATH.read_text(encoding="utf-8"))
    labels = {row["code_id"]: row["label"] for row in codebook["codes"]}
    unknown_codes = sorted(set(decisions["code_id"]) - set(labels))
    if unknown_codes:
        raise ValueError(f"Coding decisions contain unknown code IDs: {unknown_codes}")
    counts = (
        decisions.groupby("code_id", as_index=False)
        .agg(n_assignments=("respondent_id", "size"), n_respondents=("respondent_id", "nunique"))
    )
    counts["label"] = counts["code_id"].map(labels).fillna("Unlisted code")
    counts = counts[["code_id", "label", "n_assignments", "n_respondents"]].sort_values(
        ["n_assignments", "code_id"], ascending=[False, True]
    )
    counts.to_csv(counts_path, index=False)
    return counts, decisions


def _write_evidence_matrix(
    output_dir: Path,
    item_summary: pd.DataFrame,
    single_summary: pd.DataFrame,
    theme_counts: pd.DataFrame,
) -> None:
    rows: list[dict] = []
    for row in item_summary.itertuples(index=False):
        rows.append(
            {
                "evidence_stream": "quantitative",
                "measure": "primary_likert_item_mean",
                "indicator": row.item_key,
                "value": row.mean,
                "denominator": row.n,
                "interpretive_boundary": "Post-event perception on a 1-5 response scale; not objective learning.",
            }
        )
    for row in single_summary.itertuples(index=False):
        rows.append(
            {
                "evidence_stream": "quantitative",
                "measure": row.item_key,
                "indicator": row.item_key,
                "value": row.mean,
                "denominator": row.n,
                "interpretive_boundary": "Single-item post-event perception; reported separately from the primary Likert set.",
            }
        )
    for row in theme_counts.itertuples(index=False):
        rows.append(
            {
                "evidence_stream": "qualitative",
                "measure": "coded_response_field_assignments",
                "indicator": row.code_id,
                "value": row.n_assignments,
                "denominator": row.n_respondents,
                "interpretive_boundary": "Analyst-coded reported feedback; frequency does not establish learning or generalizability.",
            }
        )
    pd.DataFrame(rows).to_csv(
        output_dir / "tables" / "mixed_methods_evidence_matrix.csv", index=False
    )


def _write_figures(summary: pd.DataFrame, distributions: pd.DataFrame, figure_dir: Path) -> None:
    sns.set_theme(style="whitegrid")
    means = summary.sort_values("mean")
    plt.figure(figsize=(10, 7))
    sns.barplot(data=means, x="mean", y="item_label", color="#2f6690")
    plt.xlim(1, 5)
    plt.xlabel("Mean normalized rating (1–5)")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(figure_dir / "likert_item_means.png", dpi=180)
    plt.close()

    long = distributions.melt(
        id_vars=["item_key", "item_label"],
        value_vars=[f"pct_{score}" for score in range(1, 6)],
        var_name="rating",
        value_name="percent",
    )
    long["rating"] = long["rating"].str.replace("pct_", "", regex=False).astype(int)
    plt.figure(figsize=(11, 8))
    sns.barplot(data=long, x="percent", y="item_label", hue="rating", palette="viridis")
    plt.xlabel("Responses (%)")
    plt.ylabel("")
    plt.legend(title="Rating", ncol=5, loc="lower right")
    plt.tight_layout()
    plt.savefig(figure_dir / "likert_distribution.png", dpi=180)
    plt.close()


def run_analysis(input_path: Path = DEFAULT_INPUT, output_dir: Path = DEFAULT_OUTPUT) -> dict:
    paths = _ensure_dirs(output_dir)
    raw = load_evaluation_csv(input_path)
    ratings = build_rating_frame(raw)
    text = build_text_frame(raw)
    quality = validate_frame(raw, ratings)
    if quality["missing_analysis_columns"] or quality["invalid_rating_cells"] or quality["out_of_range_cells"]:
        raise ValueError(f"Input validation failed: {quality}")

    flags = flag_scale_direction_rows(ratings)
    flagged_ids = flags.loc[flags["flagged"], "respondent_id"].tolist()
    normalized, audit = normalize_rating_frame(ratings, flagged_ids)
    item_summary, distributions = summarize_likert(normalized, PRIMARY_ITEM_KEYS, PRIMARY_ITEM_LABELS)
    single_summary = summarize_single_items(normalized, SUMMARY_KEYS)

    normalized.to_csv(paths["processed"] / "normalized_ratings.csv", index=False)
    text.to_csv(paths["processed"] / "deidentified_text_for_coding.csv", index=False)
    flags.to_csv(paths["tables"] / "scale_direction_flags.csv", index=False)
    audit.to_csv(paths["tables"] / "normalization_audit.csv", index=False)
    item_summary.to_csv(paths["tables"] / "likert_item_summary.csv", index=False)
    distributions.to_csv(paths["tables"] / "likert_distributions.csv", index=False)
    single_summary.to_csv(paths["tables"] / "overall_satisfaction_summary.csv", index=False)
    pd.DataFrame([quality]).to_json(paths["tables"] / "data_quality_summary.json", orient="records", indent=2)
    pd.DataFrame([quality]).to_csv(paths["tables"] / "data_quality_summary.csv", index=False)
    theme_counts, theme_decisions = _theme_outputs(output_dir)
    _write_evidence_matrix(output_dir, item_summary, single_summary, theme_counts)
    extended_tables = build_extended_tables(
        raw,
        ratings,
        normalized,
        item_summary,
        distributions,
        single_summary,
        theme_counts,
        theme_decisions,
    )
    write_extended_tables(paths["tables"], extended_tables)
    extended_tables = add_derived_plot_inputs(extended_tables, normalized, distributions)
    write_extended_figures(extended_tables, paths["figures"])

    manifest = {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "input_file": str(input_path),
        "input_sha256": hashlib.sha256(input_path.read_bytes()).hexdigest(),
        "rows": int(len(raw)),
        "flagged_rows": len(flagged_ids),
        "flagged_respondent_ids": flagged_ids,
        "normalization_scope": "13 primary Likert items only; overall rating and satisfaction retained as anchors",
        "outputs": sorted(str(path.relative_to(output_dir)) for path in output_dir.rglob("*") if path.is_file()),
    }
    (output_dir / "analysis_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


def main() -> None:
    parser = argparse.ArgumentParser(description="Run the webinar evaluation analysis pipeline.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT)
    args = parser.parse_args()
    manifest = run_analysis(args.input, args.output_dir)
    print(json.dumps(manifest, indent=2))


if __name__ == "__main__":
    main()


```